# 00c_prepare_binary_crossdomain_resources.ipynb

Edit the CONFIG block, then Run All on Kaggle. This notebook does not perform face detection.

In [ ]:
# CONFIG: edit mounted paths before running on Kaggle.
V1_RESOURCE_DIR = '/kaggle/input/.../minifasnet_full_resources_v2'
CELEBA_ROOT = '/kaggle/input/.../CelebA_Spoof'
OUTPUT_ROOT = '/kaggle/working/minifasnet_binary_crossdomain_resources_v2'
MINI_DATA_SEED = 43
PARENT_SHA = 'aa67032a79627636d967dc885202540a34410c2b4d64522a5604d3aedff85b2d'
DETECTOR_SHA = '5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a'
CONTRACT_TEXT = '# PAD data and evaluation contract v2: binary cross-domain\n\nStatus: frozen pilot specification, 2026-10-02. Scope: `binary_crossdomain_v2/` only. The parent `PAD_DATA_AND_EVALUATION_CONTRACT_v1.md` remains the contract for historical v1 notebooks. Parent SHA256: `aa67032a79627636d967dc885202540a34410c2b4d64522a5604d3aedff85b2d`.\n\n## Localization and model input\n\nUse only the frozen 00-v2 SCRFD-500M raw bbox caches. Detector SHA256 `5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a`, confidence 0.5, sizes 640, 480, 320, CUDA execution provenance. A scorable record has `status == VALID`, `usable_for_pad == True`, `bbox_origin == SCRFD`, `bbox_source` beginning `SCRFD`, and a finite, positive raw `bbox_xyxy`. No GT bbox fallback or new localization. Train hygiene rejects unscorable records and native minimum bbox side below 48 px, then enforces the inherited total, per-class, class-gap, and suspicious-rate guardrails. Retain failed candidates in source and target audits.\n\nUse raw SCRFD bbox → optional Train jitter (20% chance, 0.95–1.05 scale, ±0.05 face-size center shift) → MiniFASNet 2.7× crop → 80×80 BGR float32 [0,1]. Evaluation has no jitter or augmentation. CSMR is training-only; inference has no FFT, spectral branch, selector, or margin loss.\n\n## Labels and objectives\n\nThe v2 primary label is 0 Real and 1 Attack, where Attack combines historical 1 Physical and 2 Digital. Retain the original three-class labels for stratification and subgroup diagnostics. Copied v2 models emit `[z_real,z_attack]` and security score `d=z_real-z_attack` (higher is more Real). Historical comparators A/B emit three logits and use `d=z_real-logsumexp(z_physical,z_digital)`.\n\nCelebA metadata supplies spoof type `int(labels[40])` in 0–10, lighting `int(labels[41])` in 0–4, and 40 binary attributes `labels[0:40]`. The notebook-local training-only heads are Linear(128,11), Linear(128,5), and Linear(128,40). For clean samples, `L_base=L_PAD+0.1 L_spoof+0.1 L_lighting+L_attr`, with binary CE, CE, CE, and BCEWithLogits respectively. Attributes contribute only for Real samples; an all-Attack batch uses differentiable zero.\n\nCSMR E/F use warm-up epoch 1 with `L_base`, excluded from final checkpoint selection and patience. Active CSMR uses the historical radial masks and BN-safe candidate selection, but binary aligned margin: Real `m=d`, Attack `m=-d`; select the smallest candidate margin. The active objective is `0.5 L_PAD_clean + 0.5 L_PAD_worst + 0.2 L_margin + 0.1 L_spoof_clean + 0.1 L_lighting_clean + L_attr_clean`. `L_margin` is mean SmoothL1 from worst aligned margin to detached clean aligned margin, masked to harmful drops. Auxiliary losses use clean views only. Fixed CSMR: centers [0.15,0.45,0.75], sigma 0.10, gain [0.65,0.90], DC preserved, selection chunk 32.\n\n## Pilot: six frozen runs\n\nMini Train10K samples from v1 `final_train_full.csv`; Mini Val3K samples from v1 `final_val50k.csv`. Seed 43; deterministic stratification by original three-class label; exact same ordered manifests for all runs; no sample or subject overlap. Training seed 100. A is legacy pretrained 3-logit AdamW; B is legacy scratch 3-logit AdamW; C and D are copied binary+auxiliary pretrained/scratch; E and F are C/D plus CSMR. A/B use historical 12 epochs, earliest stop 5, patience 3, label smoothing 0.1 and mild augmentation. C/D/E/F use the literature-adapted, compute-compressed recipe: SGD lr 0.005, momentum 0.9, weight decay 5e-4, batch 256, 20 epochs, MultiStepLR milestones [6,14], gamma 0.2, earliest stop 8, patience 4, no label smoothing/weighted sampling/mixup/cutmix. Copied augmentation: horizontal flip 0.5, ISO noise color shift (0.15,0.35), intensity (0.2,0.5), p 0.2; brightness/contrast limits 0.2, p 0.3; MotionBlur limit 5, p 0.2. Keep MiniFAS geometry and BGR [0,1].\n\nCheckpoint selection uses minimum Mini-Val3K ACER, tie highest AUC, with explicit no-improvement counter. Threshold is minimum-ACER Mini-Val3K threshold only. Freeze checkpoint and threshold before one official CelebA Test and one official LCC evaluation per run. Do not use Test/LCC to choose any model, threshold, optimizer, schedule, augmentation, gain or lambda. LCC is pilot/pre-scale cross-domain evidence; primary pilot metrics are AUC and EER. Report AUC, EER, TPR@FPR1%, APCER, BPCER, ACER/HTER, binary accuracy, locked threshold, candidate_n, scored_n and detector coverage. Preserve all 67,170 official CelebA Test and 7,580 official LCC candidates (314 Real, 7,266 Spoof), including unscorable rows; metrics use scored rows only.\n\nC vs E and D vs F isolate CSMR. A/B vs C/D compare training recipes with several simultaneous changes and are not single-factor causal ablations. Early stopping can prevent reaching the second LR decay; report this explicitly. No numeric result is implied by this contract.\n\n## Future full cross-domain direction\n\nReconstruct eligible full Train from **all** official CelebA Train keys plus frozen 00-v2 records; do not concatenate v1 Train+Val, reserve v1 Val subjects, or exclude Train subjects solely because they appear in official Test. Audit Train/Test identity overlap. Official CelebA Test is source development for future checkpoint/threshold selection; official LCC is primary cross-domain test. This is a distinct protocol from the Mini pilot, whose source development set is Mini Val3K.\n\n## Provenance and change control\n\n00c verifies parent contract, metadata, detector/cache hashes and frozen CUDA provenance, then exports this exact v2 contract and ordered manifest fingerprints. 01 verifies those artifacts plus model source/checkpoint hashes and stores per-run configuration and resume hashes. A mismatch fails loudly. Any change to data, architecture, preprocessing, thresholds, splits, metrics, augmentation, or schedule requires a new explicit contract version. Kaggle runtime execution and resulting metrics must be verified from actual outputs.\n'
README_TEXT = "# Binary cross-domain MiniFASNetV2 pilot\n\nThis package creates a new v2 protocol inside this directory only. Historical v1 contracts, notebooks, and outputs remain separate. The question is whether a literature-adapted binary PAD and auxiliary recipe improves the MiniFASNetV2 pilot, and whether CSMR adds cross-domain robustness to that recipe.\n\n## Execution order on Kaggle\n\n1. Use the completed `00-v2_prepare_minifasnet_full_resources.ipynb` output with frozen CUDA SCRFD bbox caches. Its provenance must match the parent contract SHA `aa67032a79627636d967dc885202540a34410c2b4d64522a5604d3aedff85b2d`.\n2. Attach the v1 resource bundle and raw CelebA-Spoof. Edit the v1 resource and CelebA paths in the CONFIG cell of `00c_prepare_binary_crossdomain_resources.ipynb`, then run it. The contract and README are embedded in 00c and exported automatically. It does **not** run SCRFD. It verifies metadata/cache hashes and produces `/kaggle/working/minifasnet_binary_crossdomain_resources_v2.zip` with a FileLink.\n3. Upload or mount the unzipped `minifasnet_binary_crossdomain_resources_v2` bundle as a Kaggle Dataset.\n4. Attach raw CelebA and official LCC images, the MiniFASNet source, and official `2.7_80x80_MiniFASNetV2.pth`. Edit the CONFIG cell of `01_mini_ablation_binary_crossdomain_10k.ipynb`, select `RUN_FILTER` if needed, then run it. It does **not** run SCRFD. Use a GPU and the frozen batch 256 for C/D/E/F; an OOM stops the run. Changing copied batch size requires a new, consistently restarted C/D/E/F protocol.\n5. Inspect all six run directories, the comparison summary, saved predictions, provenance, and checkpoint/threshold history. Kaggle execution is required before reporting measured results.\n6. Only after this pilot, create separate future full-data Vanilla and CSMR notebooks under the full cross-domain protocol.\n\n00c emits exactly:\n\n```text\nminifasnet_binary_crossdomain_resources_v2/\n  PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md\n  README.md\n  config.json\n  provenance.json\n  manifest_fingerprints.json\n  bbox/celeba_bbox_cache_full.json\n  bbox/lcc_official_evaluation_bbox_cache.json\n  labels/celeba_auxiliary_labels.csv\n  manifests/mini_train10k.csv\n  manifests/mini_val3k.csv\n  manifests/crossdomain_train_full.csv\n  manifests/crossdomain_celeba_source_dev_full.csv\n  manifests/lcc_official_evaluation_full.csv\n```\n\nThe six runs are A/B historical three-logit pretrained/scratch, C/D binary+auxiliary pretrained/scratch, and E/F C/D plus CSMR. Mini Train10K and Mini Val3K are shared identically. Mini Val3K alone selects checkpoints and calibrates thresholds. Official CelebA Test and LCC are evaluated once after each run is frozen. Mini LCC is pilot evidence only. LCC never selects checkpoints, thresholds, or hyperparameters. C vs E and D vs F are the controlled CSMR pairs; A/B vs C/D change multiple training factors.\n\nThe copied recipe is literature-adapted and compute-compressed: 20 epochs with milestones [6,14] instead of the source's 71 epochs and [20,50]. The source repository is [Assessing-Efficient-FAS-CVPR2024](https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024), especially its [config](https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/main/configs/config_large_075.py), [training](https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/main/train.py), and [CelebA loader](https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/main/datasets/celeba_spoof.py). This package keeps the project MiniFASNet architecture, SCRFD boxes, MiniFAS crop, BGR input, and candidate accounting.\n\nThe run notebook steps `MultiStepLR` after each completed epoch; milestones 6 and 14 therefore change the LR used by epochs 7 and 15. Early stopping may end before the second decay; the report records that. Resuming requires an exact per-run configuration hash and preserved RNG, optimizer, scheduler, scaler, best checkpoint, and patience state.\n\nAlbumentations has both legacy `*_limit` and current `*_range` APIs. Notebook 01 checks the installed signatures, uses the frozen numeric ranges, records the version and API variant, and fails on an unknown API. ISO noise is applied in RGB space; its output is converted back to BGR before the MiniFASNet tensor. See the [current ISO noise API](https://albumentations.ai/docs/api-reference/albumentations/augmentations/pixel/noise/).\n"

import hashlib, json, math, shutil, zipfile
from collections import Counter
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import FileLink, display

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def ordered_sha(keys): return hashlib.sha256('\n'.join(map(str,keys)).encode()).hexdigest()
def read_json(path): return json.loads(Path(path).read_text(),parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))
def write_json(path,value): Path(path).write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def subject_id(key):
    parts=PurePosixPath(str(key).replace('\\','/')).parts
    for i,p in enumerate(parts[:-1]):
        if p.lower() in {'train','test'}: return parts[i+1]
    raise ValueError('Subject cannot be parsed: '+str(key))
def label_of(meta_value):
    # Exact 00-v2 mapping, copied from its label_of implementation.
    if len(meta_value)<=40: raise ValueError('Short metadata entry')
    raw=int(meta_value[40])
    if raw==0: return 0
    if raw in [1,2,3,4,5,6,7]: return 1
    return 2
def valid_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD': return False
    if rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'): return False
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec: return False
    x1,y1,x2,y2=map(float,box)
    return bool(np.isfinite([x1,y1,x2,y2]).all() and x2>x1 and y2>y1)
def validate_record(rec,key):
    if rec.get('status') not in {'VALID','DET_FAIL','SUSPICIOUS','UNREADABLE','INVALID_BBOX'}: raise RuntimeError('Unknown status: '+key)
    if rec['status']=='VALID':
        if not valid_record(rec,key): raise RuntimeError('Invalid raw SCRFD record: '+key)
    elif rec.get('source_key')!=key or rec.get('bbox_origin')!='NONE' or rec.get('usable_for_pad') is not False or rec.get('bbox_xyxy') is not None:
        raise RuntimeError('Non-SCRFD fallback/invalid record: '+key)
def train_filter_reason(rec,key):
    if rec['status'] in {'SUSPICIOUS','DET_FAIL','UNREADABLE'}: return rec['status']
    if not valid_record(rec,key): return 'INVALID_BBOX'
    x1,y1,x2,y2=map(float,rec['bbox_xyxy'])
    if min(x2-x1,y2-y1)<48: return 'TOO_SMALL'
    return None
def frame_of(keys,meta,split,records,seed=None):
    return pd.DataFrame([{'sample_key':k,'three_class_label':label_of(meta[k]),'binary_label':int(label_of(meta[k])!=0),
      'attack_code':int(meta[k][40]),'official_split':split,'sampling_seed':seed,'subject_id':subject_id(k),
      'localization_status':records[k]['status'],'usable_for_pad':bool(records[k]['usable_for_pad']),
      'bbox_origin':records[k]['bbox_origin']} for k in keys])
def fingerprint(path,frame):
    return {'file_sha256':sha(path),'sample_count':len(frame),'ordered_sample_key_sha256':ordered_sha(frame.sample_key),
      'class_counts':{str(k):int(v) for k,v in frame.three_class_label.value_counts().sort_index().items()},
      'binary_counts':{str(k):int(v) for k,v in frame.binary_label.value_counts().sort_index().items()},
      'subject_count':int(frame.subject_id.nunique())}

def main():
    parent=Path(V1_RESOURCE_DIR); celeba=Path(CELEBA_ROOT); out=Path(OUTPUT_ROOT)
    expected=['config.json','provenance.json','PAD_DATA_AND_EVALUATION_CONTRACT_v1.md','bbox/celeba_bbox_cache_full.json',
      'bbox/lcc_official_evaluation_bbox_cache.json','manifests/final_train_full.csv','manifests/final_val50k.csv',
      'manifests/final_test_full.csv','manifests/lcc_official_evaluation_full.csv']
    for p in [*(parent/x for x in expected),celeba/'metas/intra_test/train_label.json',celeba/'metas/intra_test/test_label.json']:
        if not p.is_file(): raise FileNotFoundError(p)
    if sha(parent/'PAD_DATA_AND_EVALUATION_CONTRACT_v1.md')!=PARENT_SHA: raise RuntimeError('Parent contract SHA mismatch')
    cfg=read_json(parent/'config.json'); prov=read_json(parent/'provenance.json')
    if cfg['evaluation_contract_sha256']!=PARENT_SHA or prov['evaluation_contract_sha256']!=PARENT_SHA: raise RuntimeError('Parent provenance contract mismatch')
    if prov['scrfd_model_sha256']!=DETECTOR_SHA or prov['source_hashes']['scrfd_model']!=DETECTOR_SHA: raise RuntimeError('Detector SHA mismatch')
    if prov['scrfd_execution_provider']!='CUDAExecutionProvider' or prov['detector_threshold']!=0.5 or prov['detector_sizes']!=[640,480,320]: raise RuntimeError('Detector policy mismatch')
    if cfg['input']!={'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'}: raise RuntimeError('Input preprocessing mismatch')
    for policy in ('celeba_bbox_policy','lcc_policy'):
        if cfg[policy].get('fallback')!='none' or cfg[policy].get('provider')!=['CUDAExecutionProvider']: raise RuntimeError('Non-frozen cache policy')
    paths={'train':celeba/'metas/intra_test/train_label.json','test':celeba/'metas/intra_test/test_label.json'}
    for split,p in paths.items():
        if sha(p)!=prov['source_hashes'][split+'_label_json']: raise RuntimeError(split+' metadata SHA mismatch')
    meta={split:read_json(p) for split,p in paths.items()}
    bbox_paths={'celeba':parent/'bbox/celeba_bbox_cache_full.json','lcc':parent/'bbox/lcc_official_evaluation_bbox_cache.json'}
    for kind,p in bbox_paths.items():
        if sha(p)!=prov[kind+'_bbox_sha256']: raise RuntimeError(kind+' cache SHA mismatch')
    cache={kind:read_json(p) for kind,p in bbox_paths.items()}
    for payload in cache.values():
        header=payload.get('header',{})
        if header.get('provider')!='CUDAExecutionProvider' or header.get('detector_sha256')!=DETECTOR_SHA or header.get('contract_sha256')!=PARENT_SHA: raise RuntimeError('Cache header mismatch')
    records=cache['celeba']['records']; lcc_records=cache['lcc']['records']
    if set(records)!=set(meta['train'])|set(meta['test']): raise RuntimeError('CelebA cache key coverage mismatch')
    for k,r in records.items(): validate_record(r,k)
    for k,r in lcc_records.items(): validate_record(r,k)
    v1_fp=prov['manifest_fingerprints']
    def old_frame(name):
        p=parent/'manifests'/(name+'.csv'); f=pd.read_csv(p,keep_default_na=False)
        if sha(p)!=v1_fp[name]['file_sha256'] or ordered_sha(f.sample_key)!=v1_fp[name]['ordered_sample_key_sha256']: raise RuntimeError(name+' fingerprint mismatch')
        if f.sample_key.duplicated().any(): raise RuntimeError(name+' duplicate key')
        return f
    old_train=old_frame('final_train_full'); old_val=old_frame('final_val50k'); old_test=old_frame('final_test_full')
    lcc_path=parent/'manifests/lcc_official_evaluation_full.csv'; lcc=pd.read_csv(lcc_path,keep_default_na=False)
    if sha(lcc_path)!=prov['lcc_manifest_sha256'] or len(lcc)!=7580 or int((lcc.label==0).sum())!=314 or int((lcc.label==1).sum())!=7266: raise RuntimeError('LCC population/hash mismatch')
    if len(old_test)!=67170 or set(old_test.sample_key)!=set(meta['test']): raise RuntimeError('Official Test population mismatch')
    for f in (old_train,old_val):
        for row in f.itertuples(index=False):
            if not valid_record(records[str(row.sample_key)],str(row.sample_key)): raise RuntimeError('Mini source has invalid bbox')
    if set(old_train.subject_id.astype(str)) & set(old_val.subject_id.astype(str)): raise RuntimeError('Parent subject leakage')
    def sample(frame,n):
        if len(frame)<n or set(frame.three_class_label.astype(int))!={0,1,2}: raise RuntimeError('Mini pool insufficient')
        selected,_=train_test_split(np.arange(len(frame)),train_size=n,stratify=frame.three_class_label.astype(int),random_state=MINI_DATA_SEED)
        return frame.iloc[sorted(selected)].sample_key.astype(str).tolist()
    mini_train=frame_of(sample(old_train,10000),meta['train'],'train',records,MINI_DATA_SEED)
    mini_val=frame_of(sample(old_val,3000),meta['train'],'train',records,MINI_DATA_SEED)
    if set(mini_train.sample_key)&set(mini_val.sample_key) or set(mini_train.subject_id)&set(mini_val.subject_id): raise RuntimeError('Mini split leakage')
    for f,n in ((mini_train,10000),(mini_val,3000)):
        if len(f)!=n or f.sample_key.duplicated().any() or set(f.three_class_label)!={0,1,2} or not f.usable_for_pad.all() or set(f.bbox_origin)!={'SCRFD'}: raise RuntimeError('Mini split invariant failed')
    # Reconstruct all eligible official Train, intentionally without v1 subject exclusions.
    train_keys=list(meta['train']); reasons={k:train_filter_reason(records[k],k) for k in train_keys}
    eligible=[k for k in train_keys if reasons[k] is None]
    candidate=Counter(label_of(meta['train'][k]) for k in train_keys); dropped=Counter(label_of(meta['train'][k]) for k in train_keys if reasons[k] is not None)
    rates={str(c):dropped[c]/candidate[c] for c in range(3)}; total=len(train_keys)-len(eligible)
    suspicious=sum(records[k]['status']=='SUSPICIOUS' for k in train_keys)/len(train_keys)
    if total/len(train_keys)>.05 or max(rates.values())>.08 or max(rates.values())-min(rates.values())>.03 or suspicious>.01: raise RuntimeError('Inherited Train hygiene guardrail failed')
    full_train=frame_of(eligible,meta['train'],'train',records)
    source_dev=frame_of(list(meta['test']),meta['test'],'test',records)
    if len(source_dev)!=67170: raise RuntimeError('Source-dev candidate count changed')
    if set(full_train.sample_key)&set(source_dev.sample_key): raise RuntimeError('Official split sample leakage')
    if set(lcc.path.astype(str))!=set(lcc_records): raise RuntimeError('LCC cache coverage mismatch')
    for row in lcc.itertuples(index=False):
        r=lcc_records[str(row.path)]
        if str(row.status)!=r['status'] or bool(row.usable_for_pad)!=bool(r['usable_for_pad']) or str(row.bbox_origin)!=r['bbox_origin']: raise RuntimeError('LCC manifest/cache mismatch')
    attrs=[f'attr_{i:02d}' for i in range(40)]; aux_rows=[]
    for split,entries in meta.items():
        for k,labels in entries.items():
            if len(labels)<42: raise RuntimeError('Short CelebA labels: '+k)
            a=list(map(int,labels[:40])); spoof=int(labels[40]); lighting=int(labels[41]); cls=label_of(labels)
            if len(a)!=40 or set(a)-{0,1} or not 0<=spoof<11 or not 0<=lighting<5: raise RuntimeError('Aux label range: '+k)
            aux_rows.append({'sample_key':k,'official_split':split,'three_class_label':cls,'binary_label':int(cls!=0),'spoof_type':spoof,'lighting':lighting,**dict(zip(attrs,a))})
    aux=pd.DataFrame(aux_rows)
    if aux.sample_key.duplicated().any() or set(aux.binary_label)-{0,1}: raise RuntimeError('Aux key/label invariant')
    for d in ('bbox','labels','manifests'): (out/d).mkdir(parents=True,exist_ok=True)
    manifests={'mini_train10k':mini_train,'mini_val3k':mini_val,'crossdomain_train_full':full_train,'crossdomain_celeba_source_dev_full':source_dev}
    fps={}
    for name,f in manifests.items():
        p=out/'manifests'/(name+'.csv'); f.to_csv(p,index=False); fps[name]=fingerprint(p,f)
    aux_path=out/'labels/celeba_auxiliary_labels.csv'; aux.to_csv(aux_path,index=False)
    shutil.copy2(lcc_path,out/'manifests/lcc_official_evaluation_full.csv')
    for kind,p in bbox_paths.items():
        dst=out/'bbox'/p.name; shutil.copy2(p,dst)
        if sha(dst)!=sha(p): raise RuntimeError('BBox copy drift')
    contract_sha=hashlib.sha256(CONTRACT_TEXT.encode()).hexdigest()
    if contract_sha!='2f1f6eb6351550d491e4fb59e2151b1ccd9b000c0985cd32f12621f9d58c5f91':raise RuntimeError('Embedded v2 contract SHA mismatch')
    contract_path=out/'PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md'
    contract_path.write_text(CONTRACT_TEXT);(out/'README.md').write_text(README_TEXT)
    if sha(contract_path)!=contract_sha:raise RuntimeError('Exported v2 contract SHA mismatch')
    lcc_fp={'file_sha256':sha(out/'manifests/lcc_official_evaluation_full.csv'),'candidate_n':len(lcc),'scored_n':int(lcc.usable_for_pad.sum()),'ordered_path_sha256':ordered_sha(lcc.path)}
    fps['lcc_official_evaluation_full']=lcc_fp
    audit={'train_candidate_n':len(train_keys),'train_scored_n':len(full_train),'train_subject_n':int(full_train.subject_id.nunique()),
      'train_class_counts':fps['crossdomain_train_full']['class_counts'],'train_attack_code_counts':{str(k):int(v) for k,v in full_train.attack_code.value_counts().items()},
      'train_filter_reason_counts':dict(Counter(x for x in reasons.values() if x)),'train_filter_rates':rates,
      'source_dev_candidate_n':len(source_dev),'source_dev_scored_n':int(source_dev.usable_for_pad.sum()),
      'source_dev_detector_coverage':float(source_dev.usable_for_pad.mean()),
      'train_test_identity_overlap_n':len(set(full_train.subject_id)&set(source_dev.subject_id)),
      'lcc_candidate_n':len(lcc),'lcc_scored_n':lcc_fp['scored_n'],'lcc_detector_coverage':lcc_fp['scored_n']/len(lcc),
      'lcc_real_n':314,'lcc_attack_n':7266}
    config={'schema':'binary_crossdomain_resources_v2','parent_contract_sha256':PARENT_SHA,'v2_contract_sha256':contract_sha,
      'mini_data_seed':MINI_DATA_SEED,'input':cfg['input'],'detector_sha256':DETECTOR_SHA,'audit':audit}
    provenance={'schema':'binary_crossdomain_resources_v2','parent_contract_sha256':PARENT_SHA,'v2_contract_sha256':contract_sha,
      'source_hashes':prov['source_hashes'],'celeba_bbox_sha256':sha(bbox_paths['celeba']),'lcc_bbox_sha256':sha(bbox_paths['lcc']),
      'detector_sha256':DETECTOR_SHA,'scrfd_execution_provider':'CUDAExecutionProvider','mini_data_seed':MINI_DATA_SEED,
      'auxiliary_labels_sha256':sha(aux_path),'manifest_fingerprints':fps,'audit':audit}
    write_json(out/'config.json',config); write_json(out/'provenance.json',provenance); write_json(out/'manifest_fingerprints.json',fps)
    zip_path=Path(str(out)+'.zip')
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
        for path in sorted(out.rglob('*')):
            if path.is_file(): z.write(path,Path(out.name)/path.relative_to(out))
    print('Audit:',json.dumps(audit,indent=2));print('ZIP:',zip_path); display(FileLink(str(zip_path)))
main()
